# pyskills API
> API details

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
from fastcore.xml import Safe
from fastcore.docments import MarkdownRenderer,can_render
from fastcore.xdg import *
from importlib.metadata import entry_points
from inspect import signature
import importlib.util, types, inspect, collections, ast, site

## Overview

A plugin system allowing Python packages to register "skills" — units of LLM-usable functionality — via standard Python entry points. An LLM host (e.g. solveit) discovers available pyskills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen pyskills into context.

### Entry Point Convention

Packages register pyskills under the group `pyskills`:

```toml
[project.entry-points.pyskills]
my_skill = "mypackage.skill"
```

The value is a module path (no `:attribute` needed). The module's docstring first paragraph serves as the pyskill description.

### Skill Module Contract

A pyskill module MUST have:

- **Docstring** — first paragraph is the short description shown to the LLM for pyskill selection; remaining will be read by the LLM to get full details on the pyskill.

A pyskill module MAY have:

- **`__all__`** — the available symbols imported.

### Discovery API

```python
def list_pyskills() -> dict[str, str]
```
Returns `{name: description}` for all registered pyskills, using `find_spec` + AST parsing — no imports.

```python
import mypackage.skill
```
Standard python native import

```python
doc(mypackage.skill)       # module overview: classes, functions, submodules, allows
doc(SomeClass)             # class detail: bases, __init__, methods, properties
doc(some_func)             # function detail: full signature with docments
xdir(mypackage.skill)      # filtered names for public symbols
```
Inspect at increasing detail — works on any Python module, not just pyskills.


### Host Integration

The host (e.g. solveit, claude code, codex, ...) would:

1. Call `list_pyskills()` at startup to build a pyskill catalogue
2. Include the list with each prompt
3. Call `import {module}` followed by `doc({module})` for chosen pyskills

## Listing pyskills

In [ ]:
ep = entry_points()
es = first(ep.select(group='pyskills', name='pyskills.skill'))
es

EntryPoint(name='pyskills.skill', value='pyskills.skill', group='pyskills')

In [ ]:
#| export
def ep_desc(ep):
    "First paragraph of docstring for entry point `ep`, without importing it"
    spec = importlib.util.find_spec(ep.value.split(':')[0])
    if not spec or not spec.origin: return None
    tree = ast.parse(Path(spec.origin).read_text())
    doc = ast.get_docstring(tree)
    if not doc: return None
    return doc.split('\n\n')[0]

In [ ]:
print(ep_desc(es))

Pyskills is a plugin system allowing Python packages to register "skills" (units of LLM-usable functionality) via standard Python entry points. An LLM host (e.g. solveit) discovers available pyskills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen pyskills into context using standard imports.


In [ ]:
#| export
@llmtool
def list_pyskills():
    """Returns {module: description} for all pyskills. To load a module, use `import {module}` then view `doc({module}).
    **NB**: pyskills are *THE* critical way to extend functionality. *ALWAYS* check for pyskills to help with tasks.
    If unsure whether a particular pyskill might help, load it and grabs its docs to see!"""
    return {ep.value: ep_desc(ep) for ep in entry_points().select(group='pyskills')}

In [ ]:
list_pyskills()

{'pyskills.edit': 'Text editing tools for modifying files and ipynb notebook cells. Each operation — insert, replace, delete lines, and string replacement — has both a `file_*` and `cell_*` variant. All return unified diffs showing what changed, or `"none: No changes."`, or `"error: ..."`.',
 'pyskills.skill': 'Pyskills is a plugin system allowing Python packages to register "skills" (units of LLM-usable functionality) via standard Python entry points. An LLM host (e.g. solveit) discovers available pyskills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen pyskills into context using standard imports.',
 'test.skill': 'A test skill.'}

## `allow`

In [ ]:
#| export
__pytools__ = collections.defaultdict(set)

In [ ]:
#| export
_all_ = ['__pytools__']

In [ ]:
#| export
def allow(*c, allow_policy=None): # Callable that raises if call not allowed
    "Add all items in `c` to `__pytools__`, optionally constrained by `allow_policy`"
    def _wrap(v):
        if allow_policy is None or v is ... or isinstance(v, tuple): return v
        return (v, allow_policy)
    for o in c:
        if isinstance(o, dict):
            for k,v in o.items(): __pytools__[k].update(_wrap(x) for x in listify(v))
        else:
            objclass = getattr(o, '__objclass__', None)
            if objclass is not None:
                __pytools__[objclass].add(_wrap(o.__name__))
                continue
            qualname = getattr(o, '__qualname__', '') or ''
            mod = sys.modules.get(getattr(o, '__module__', '__main__'), sys.modules.get('__main__'))
            if '.' in qualname:
                cls = getattr(mod, qualname.rsplit('.', 1)[0], None)
                if cls is not None:
                    __pytools__[cls].add(_wrap(o.__name__))
                    continue
            __pytools__[mod].add(_wrap(o.__name__))
    if len(c)==1 and callable(c[0]): return c[0]

`__pytools__` is a `defaultdict(set)` mapping classes/modules to their allowed method/function names (or `...` for all public methods). Values can be plain strings or `(name, AllowPolicy)` tuples for allow-checked methods. `allow` registers entries — callables are added under their module, dicts go directly into `__pytools__`.

In [ ]:
def _test_fn(): pass
_test_fn.__module__ = '__main__'
_test_fn.__name__ = 'my_test_func'
allow(_test_fn)
assert 'my_test_func' in __pytools__[sys.modules['__main__']]
allow({str: ['zfill']})
assert 'zfill' in __pytools__[str]
allow({list: ...})
assert ... in __pytools__[list]
__pytools__[sys.modules['__main__']].discard('my_test_func')


allow(collections.Counter.most_common)
assert 'most_common' in __pytools__[collections.Counter]
__pytools__[collections.Counter].discard('most_common')

### Allow policies

In [ ]:
#| export
def chk_dest(p, ok_dests):
    resolved = str(Path(p).resolve())
    if not any(resolved == (rd := str(Path(d).expanduser().resolve())) or resolved.startswith(rd + '/') for d in ok_dests):
        raise PermissionError(f"Dest '{p}' not allowed; permitted: {ok_dests}")

`chk_dest` resolves a path and verifies it falls under one of the allowed destination prefixes. Raises `PermissionError` if not. Used by all `AllowPolicy` subclasses.

In [ ]:
chk_dest('/tmp/foo.txt', ['/tmp'])
try: chk_dest('/etc/passwd', ['/tmp'])
except PermissionError: print("Correctly blocked /etc/passwd")

Correctly blocked /etc/passwd


In [ ]:
#| export
class AllowPolicy:
    "Base for allow destination policies"
    def __call__(self, obj, args, kwargs, ok_dests): raise NotImplementedError

class PosAllowPolicy(AllowPolicy):
    "Check positional/keyword arg is an allowed destination"
    def __init__(self, pos=0, kw=None): store_attr()
    def __call__(self, obj, args, kwargs, ok_dests):
        p = kwargs.get(self.kw) if self.kw and self.kw in kwargs else args[self.pos] if self.pos < len(args) else None
        if p is not None: chk_dest(p, ok_dests)

class PathWritePolicy(AllowPolicy):
    "Check resolved Path self, optionally also target args"
    def __init__(self, target_pos=None, target_kw=None): store_attr()
    def __call__(self, obj, args, kwargs, ok_dests):
        chk_dest(obj, ok_dests)
        if self.target_pos is not None and self.target_pos < len(args): chk_dest(args[self.target_pos], ok_dests)
        if self.target_kw and self.target_kw in kwargs: chk_dest(kwargs[self.target_kw], ok_dests)

class OpenWritePolicy(AllowPolicy):
    "Check open() only when mode is writable"
    def __call__(self, obj, args, kwargs, ok_dests):
        mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
        if any(c in mode for c in 'wax+'): chk_dest(args[0] if args else kwargs.get('file'), ok_dests)

Three `AllowPolicy` subclasses handle different allow-checking patterns.

In [ ]:
pp = PosAllowPolicy(1, 'dst')
pp(None, ['src', '/tmp/ok'], {}, ['/tmp'])
try: pp(None, ['src', '/root/bad'], {}, ['/tmp'])
except PermissionError: print("PosAllowPolicy blocked /root/bad")

pwp = PathWritePolicy()
pwp(Path('/tmp/f.txt'), [], {}, ['/tmp'])
try: pwp(Path('/etc/f.txt'), [], {}, ['/tmp'])
except PermissionError: print("PathWritePolicy blocked /etc/f.txt")

owp = OpenWritePolicy()
owp(None, ['/tmp/f.txt', 'w'], {}, ['/tmp'])
owp(None, ['/etc/passwd', 'r'], {}, ['/tmp'])
try: owp(None, ['/root/f.txt', 'w'], {}, ['/tmp'])
except PermissionError: print("OpenWritePolicy blocked write to /root/f.txt")

PosAllowPolicy blocked /root/bad
PathWritePolicy blocked /etc/f.txt
OpenWritePolicy blocked write to /root/f.txt


Allow policies are stored as `(name, AllowPolicy)` tuples directly inside `__pytools__` sets.

## doc / xdir

In [ ]:
import pyskills.skill

In [ ]:
#| export
def _is_own(sym, n):
    "Whether name `n` in module `sym` is an owned symbol or sibling submodule"
    m = getattr(sym, n, None)
    if m is None: return False
    if getattr(m, '__module__', None) == sym.__name__: return True
    return isinstance(m, types.ModuleType) and m.__name__.split('.')[0] == sym.__name__.split('.')[0]

In [ ]:
assert _is_own(pyskills.skill, 'skill_test_func')
assert _is_own(pyskills.skill, 'SkillTestClass')
assert not _is_own(pyskills.skill, 'inspect')
assert not _is_own(pyskills.skill, '_private')

In [ ]:
#| export
def _imported_submods(sym):
    "Sibling submodules explicitly imported via `import pkg.sub` by module `sym`"
    pkg = sym.__name__.rsplit('.', 1)[0] if '.' in sym.__name__ else None
    if not pkg: return {}
    res = {}
    for node in ast.walk(ast.parse(inspect.getsource(sym))):
        if not isinstance(node, ast.Import): continue
        for alias in node.names:
            if alias.name.startswith(pkg + '.') and alias.name != sym.__name__:
                m = sys.modules.get(alias.name)
                if m: res[alias.asname or alias.name] = m
    return res

def _is_own(sym, n):
    "Whether name `n` in module `sym` is an owned symbol or sibling submodule"
    m = getattr(sym, n, None)
    if m is None: return False
    if getattr(m, '__module__', None) == sym.__name__: return True
    if not isinstance(m, types.ModuleType): return False
    mname = m.__name__
    if sym.__name__.startswith(mname + '.'): return False
    return mname.split('.')[0] == sym.__name__.split('.')[0]

`xdir` returns filtered `(name, obj)` pairs for a module's public symbols (respecting `__all__`, skipping private names), or a class's `__init__` and public methods. For modules, it also includes sibling submodules explicitly imported by the module.

In [ ]:
#| export
def _xdir(sym):
    "Filtered (name, obj) pairs for public symbols of a module or class (or anything with `__dir__`)"
    if isinstance(sym, types.ModuleType):
        names = getattr(sym, '__all__', None)
        if names is None: names = [n for n in sorted(dir(sym)) if not n.startswith('_') and _is_own(sym, n)]
        res = [(n, getattr(sym, n)) for n in names if getattr(sym, n, None) is not None]
        subs = _imported_submods(sym)
        return res + [(n, m) for n,m in sorted(subs.items()) if n not in dict(res)]
    if isinstance(sym, type):
        res = []
        init = getattr(sym, '__init__', None)
        if init and init is not object.__init__: res.append(('__init__', init))
        return res + [(n, v) for n,v in sorted(sym.__dict__.items()) if not n.startswith('_')]
    if '__dir__' not in type(sym).__dict__: return []
    return [(n, getattr(sym, n, None)) for n in sorted(dir(sym)) if not n.startswith('_')]

@allow
def xdir(sym):
    "Filtered names for public symbols of a module or class (or anything with `__dir__`)"
    return [o for o,_ in _xdir(sym)]

In [ ]:
xdir(pyskills.skill)

['SkillTestClass', 'skill_test_func', 'pyskills.createskill']

In [ ]:
#| export
@allow
def doc(sym)->str:
    if isinstance(sym, type): return _doc_class(sym)
    if isinstance(sym, types.ModuleType): return _doc_module(sym)
    if hasattr(sym, '_repr_markdown_'): return sym._repr_markdown_()
    if callable(sym) and can_render(sym): return Safe(MarkdownRenderer(sym))
    if (items := _xdir(sym)): return _doc_instance(sym, items)
    if '__str__' in type(sym).__dict__: return str(sym)
    if '__repr__' in type(sym).__dict__: return repr(sym)
    return Safe(MarkdownRenderer(sym))

For a function, `doc` renders the docstring and full signature with docments (parameter comments):

In [ ]:
print(doc(pyskills.skill.skill_test_func))

def skill_test_func(
    x:int=0, # the input
)->str: # the output
"""A test function"""


In [ ]:
#| export
def _fmt_method(name, method, prefix=''):
    try: sig = str(signature(method))
    except (ValueError, TypeError): sig = '(...)'
    d = getattr(method, '__doc__', None)
    res = f'    {prefix}def {name}{sig}: ...'
    if d and name != '__init__': res += f'  # {d.splitlines()[0].strip()}'
    return res

def _doc_class(sym):
    bases = ','.join(b.__name__ for b in sym.__mro__[1:-1]) if sym.__mro__[1:-1] else ''
    parts = [f'class {sym.__name__}({bases}):' if bases else f'class {sym.__name__}:']
    for name,raw in _xdir(sym):
        if isinstance(raw, property): parts.append(_fmt_method(name, raw.fget, '@property\n    '))
        elif isinstance(raw, classmethod): parts.append(_fmt_method(name, raw.__func__, '@classmethod\n    '))
        elif isinstance(raw, staticmethod): parts.append(_fmt_method(name, raw.__func__, '@staticmethod\n    '))
        elif callable(raw): parts.append(_fmt_method(name, raw))
    d = sym.__doc__ or (getattr(sym, '__init__', None) and sym.__init__.__doc__)
    if d: parts.insert(1, '    """' + inspect.cleandoc(d).replace('\n', '\n    ') + '"""')
    return parts[0] + '\n' + '\n'.join(parts[1:])


For a class, `doc` shows the class hierarchy, docstring, `__init__` signature, and all public methods/properties with their first docstring line:

In [ ]:
print(doc(pyskills.skill.SkillTestClass))

class SkillTestClass(str):
    """Some class.
    More info about it."""
    def __init__(self): ...
    def f(self, x: int = 0) -> str: ...  # A test method
    @property
    def g(self) -> str: ...  # A test prop


In [ ]:
#| export
def _fmt_allows(mod):
    src = inspect.getsource(mod)
    tree = ast.parse(src)
    lines = src.splitlines()
    calls = [node for node in ast.walk(tree) if isinstance(node, ast.Expr)
             and isinstance(node.value, ast.Call) and ast.unparse(node.value.func) == 'allow']
    if not calls: return ''
    return '\n## allows:\n' + '\n'.join(f'- {ast.unparse(node.value)}' for node in calls)

In [ ]:
#| export
def _doc_module(mod):
    parts = [f'# module {mod.__name__}:\n']
    if mod.__doc__: parts.append(f'"""{inspect.cleandoc(mod.__doc__)}\n"""')
    typs,funcs,subs = [],[],[]
    for name,obj in _xdir(mod):
        ds = getattr(obj, '__doc__', None)
        if not ds and isinstance(obj, type): ds = getattr(getattr(obj, '__init__', None), '__doc__', None)
        d = (ds or '').splitlines()
        comment = f': ...  # {d[0].strip()}' if d and d[0].strip() else ''
        if isinstance(obj, types.ModuleType): subs.append(f'  {name}{comment}')
        elif isinstance(obj, type):
            bases = ','.join(b.__name__ for b in obj.__mro__[1:-1])
            base_str = f'({bases})' if bases else ''
            typs.append(f'- class {name}{base_str}{comment}')
        elif callable(obj):
            try: sig = str(signature(obj))
            except (ValueError, TypeError): sig = '(...)'
            funcs.append(f'- def {name}{sig}{comment}')
    if typs: parts += ['\n## types:', *typs]
    if funcs: parts += ['\n## functions:', *funcs]
    if subs: parts += ['\n## submodules:', *subs]
    parts.append(_fmt_allows(mod))
    return '\n'.join(parts)

For a module, `doc` shows the docstring, all public classes and functions with their signatures and first docstring line, submodules, and any `allow()` calls:


In [ ]:
print(doc(pyskills.skill)[-450:])


## Creating pyskills

`from pyskills import createskill; doc(createskill)` for how to build and register your own pyskill modules, including the allow/policy system.
"""

## types:
- class SkillTestClass(str): ...  # Some class.

## functions:
- def skill_test_func(x: int = 0) -> str: ...  # A test function

## submodules:
  pyskills.createskill: ...  # How to create a pyskills pyskill module.

## allows:
- allow(skill_test_func, SkillTestClass)


In [ ]:
#| export
def _doc_instance(sym, items):
    parts = [f'Instance of type {type(sym).__name__}:']
    for name,obj in items:
        ds = (getattr(obj, '__doc__', None) or '').splitlines()
        comment = f'  # {ds[0].strip()}' if ds and ds[0].strip() else ''
        if callable(obj):
            try: sig = str(signature(obj))
            except (ValueError, TypeError): sig = '(...)'
            parts.append(f'- {name}{sig}{comment}')
        else: parts.append(f'- {name}: {type(obj).__name__} = {repr(obj)[:80]}')
    return '\n'.join(parts)

In [ ]:
class Pet:
    def __init__(self, name, sound):
        self.name,self.sound = name,sound
    def speak(self):
        "Make the pet's sound"
        return f'{self.name} says {self.sound}!'
    def __dir__(self): return ['name', 'sound', 'speak']

p = Pet('Rex', 'woof')
print(doc(p))

Instance of type Pet:
- name: str = 'Rex'
- sound: str = 'woof'
- speak()  # Make the pet's sound


In [ ]:
#| export
@allow
def docfind(o, q, n=2, _pre=''):
    "Search `doc()` recursively through `xdir(o)`, looking at submodules, classes, and functions, to depth `n`"
    pat = re.compile(q, re.IGNORECASE)
    res = []
    try: d = doc(o) or ''
    except: d = ''
    if pat.search(d): res.append(_pre + ' // ' + d.split('\n')[0])
    if n > 0:
        for k in xdir(o):
            try: child = getattr(o, k)
            except: continue
            nm = f'{_pre}.{k}' if _pre else k
            res += docfind(child, q, n-1, nm)
    return res

## Skills registration

Pyskills can be added as standard modules with pyproject entrypoints. But for convenience, they can also be added to a custom pyskills XDG directory, which is automatically added to sys.path.

In [ ]:
#| export
def pyskills_dir():
    "Directory for user pyskills"
    return xdg_data_home() / 'pyskills'

def ensure_pyskills_dir():
    "Create xdg pyskills dir and .pth file if needed"
    sd = str(pyskills_dir())
    if sd in sys.path: return
    Path(sd).mkdir(parents=True, exist_ok=True)
    sps = site.getsitepackages()
    try: sps.append(site.getusersitepackages())
    except AttributeError: pass
    for sp in sps:
        pth = Path(sp) / 'pyskills.pth'
        if pth.exists() and sd in pth.read_text(): break
        try:
            Path(sp).mkdir(parents=True, exist_ok=True)
            pth.write_text(sd + '\n')
            break
        except (PermissionError, OSError): pass
    sys.path.append(sd)

In [ ]:
#| export
ensure_pyskills_dir()

`pyskills_dir` returns the XDG data home path for user pyskills. `ensure_pyskills_dir` creates that directory if needed and writes a `.pth` file into `site-packages` so Python automatically adds it to `sys.path`. You can drop pyskill modules there without manual path configuration and which can be available across venvs.

In [ ]:
#| export
def clear_mod(prefix):
    "Clear modules starting with `prefix` from python caches"
    for k in list(sys.modules):
        if k.startswith(prefix): del sys.modules[k]
    importlib.invalidate_caches()

`clear_mod` purges all cached modules matching a prefix from `sys.modules` and invalidates import caches, ensuring a fresh import on next access. Used after enabling/disabling pyskills so changes take effect immediately.

In [ ]:
#| export
def _pyskill_dist(name): return pyskills_dir() / f'{name.replace(".", "_")}-0.0.1.dist-info'

def enable_pyskill(name):
    "Enable pyskill `name` by creating its dist-info entry point"
    di = _pyskill_dist(name)
    di.mkdir(exist_ok=True)
    (di / 'METADATA').write_text(f'Name: {name.replace(".", "_")}\nVersion: 0.0.1\n')
    (di / 'entry_points.txt').write_text(f'[pyskills]\n{name} = {name}\n')
    clear_mod(name.split('.')[0])

def register_pyskill(name, docstr, code=''):
    "Register a pyskill module `name` in the xdg pyskills dir"
    sd = pyskills_dir()
    parts = name.split('.')
    mod_dir = sd / Path(*parts[:-1]) if len(parts) > 1 else sd
    mod_dir.mkdir(parents=True, exist_ok=True)
    for i in range(len(parts)-1): (sd / Path(*parts[:i+1]) / '__init__.py').touch()
    (mod_dir / f'{parts[-1]}.py').write_text(f'"""{docstr}"""\n{code}')
    enable_pyskill(name)

`enable_pyskill` creates a minimal `dist-info` directory with an entry point so the pyskill is listed. `register_pyskill` also writes an actual module file (with docstring and code) into the pyskills directory, and creates any needed `__init__.py` files for nested packages.

This lets you programmatically create and register a pyskill without a full package install.

In [ ]:
#| export
def disable_pyskill(name):
    "Disable pyskill `name` by removing its dist-info entry point"
    di = _pyskill_dist(name)
    if di.exists(): shutil.rmtree(di)
    clear_mod(name.split('.')[0])

`disable_pyskill` removes the `dist-info` directory for a pyskill, so it no longer appears in entry point discovery. It also clears the module cache so the pyskill is fully unloaded.

In [ ]:
#| export
def delete_pyskill(name):
    "Delete pyskill `name` module files and dist-info"
    disable_pyskill(name)
    sd = pyskills_dir()
    parts = name.split('.')
    mod_file = sd / Path(*parts[:-1]) / f'{parts[-1]}.py' if len(parts) > 1 else sd / f'{parts[-1]}.py'
    if mod_file.exists(): mod_file.unlink()

## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()